### scRNA-seq Tutorial 2: Reference mapping

In this tutorial, you will learn how to annotate the cells in your dataset through transfering the labels from an annotated embryonic reference. We will be mapping our datasets onto post-conception week 3, 4 and 5 cells from [PMID 37192616](https://pubmed.ncbi.nlm.nih.gov/37192616/).

First, we import the packages we need to access the functions we use. If you need to use another package, install it and import it below. The main package we use to map single cell RNA-seq data is [scvi](https://docs.scvi-tools.org/en/stable/), which we install in the next cell.

In [ ]:
%pip install scvi-tools


In [ ]:
import os
import sys
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scvi
import anndata as ad

Load the datasets you wish to use for annotation and mapping. In this example, we use the day5 dataset from the first tutorial as our 'query', and we load an already annotated dataset of embryonic cells from post-conception weeks (PCW) 3, 4 and 5 as the reference. 

Beware that the software we will use, which is scVI, requires non-normalized data. For this reason, we load the filtered matrices.


In [ ]:
#Load the data you wish to annotate; In this example, we use the concatenated dataset from the first tutorial
adata = sc.read_h5ad ("data/3802619/GSE167011_IPS_lung_differentiation_day5_filtered.h5ad")
adata

For the reference to be embedded in latent space, the non-transformed read counts are used. Therefore, we read the filtered counts with the attached the cell annotation. Here we can try to map onto the post-conceptional week of choice, based on expected developmental timing, or map over all three datasets at once and letting the model decide the time. 

We provide below code for both methods. Try out both and decide which strategy you will go for. 

In [ ]:
## Read all three references and concatenate 

reference_w3 = sc.read_h5ad("data/references/PCW3_filtered_annotated.h5ad")
reference_w4 = sc.read_h5ad("data/references/PCW4_filtered_annotated.h5ad")
reference_w5 = sc.read_h5ad("data/references/PCW5_filtered_annotated.h5ad")

for reference in [reference_w3, reference_w4, reference_w5]:
    reference.var_names = reference.var_names.str.replace(r"\.\d+$", "", regex=True)
    reference.var_names_make_unique()

In [ ]:
week = 3

reference = sc.read_h5ad(f'data/references/PCW{week}_filtered_annotated.h5ad')


In [ ]:
query = sc.read_h5ad("data/3802619/GSE167011_IPS_lung_differentiation_day5_filtered.h5ad")

query.obs["cell_type"] = "Unknown"

Let's first take a look at out reference dataset and check out all the cell types that have been annotated. The annotations can be found in the ```cell_type``` column.

In [ ]:
reference.obs["cell_type"]

In order to perform the label transfer from the reference, we will train a model that projects the gene expression data from our reference into latent space. Then we map the query data into the same latent space using this trained model. Finally, we assign labels to query cells based on their proximity to labeled reference cells in this space, typically using a classifier such as k-nearest neighbors or a built-in classifier (as in scANVI).


Now we train the model on the reference

In [ ]:
#This step might take a while

scvi.model.SCANVI.setup_anndata(
    reference,
    labels_key="cell_type",
    unlabeled_category="Unknown"
)

model = scvi.model.SCANVI(reference,
                        n_layers=2,
                        encode_covariates=True,
                        deeply_inject_covariates=True,
                        use_layer_norm="both",
                        use_batch_norm="none",
)
model.train(max_epochs=50)


In [ ]:
model.save("models/pcw_all_scanvi_ref", overwrite=True, save_anndata=True)


Let's extract the coordinates of our cells in latent space.

In [ ]:
#Exract coordinates of latent space 
reference.obsm["X_scVI"] = model.get_latent_representation()

#Mark the reference for future concatenation
reference.obs["mapping_view"] = "reference"

Now we can predict the labels of our query data.

In [ ]:
#Mark the query 
query.obs["mapping_view"] = "query"

#Pad the missing gene names
query_prep= scvi.model.SCANVI.prepare_query_anndata(
    query.copy(),
    model,
    inplace=False,
)

#Get reference gene names
ref_vars = scvi.model.SCANVI.prepare_query_anndata(
    query_prep,
    model,
    return_reference_var_names=True,
)

#Keep to common genes
common = pd.Index(ref_vars).intersection(query_prep.var_names)
query_prep = query_prep[:, ref_vars].copy()

scanvi_query = scvi.model.SCANVI.load_query_data(
    query_prep,
    model,
    inplace_subset_query_vars=False,
)


scanvi_query.train(max_epochs=20, plan_kwargs={"weight_decay": 0.0})

#Extract latent representation
query.obsm["X_scVI"] = scanvi_query.get_latent_representation()

#Predict query cell type
query.obs["cell_type"] = scanvi_query.predict()

query.obs["cell_type"].head()


Let's concatenate the two adata objects for plotting.

In [ ]:
adata_combined = ad.concat(
    [reference, query],
    label="dataset",
    keys=["reference", "query"],
    join="outer",
    index_unique="-"
)

# if you want to be explicit:
adata_combined.obs["mapping_view"] = adata_combined.obs["dataset"].astype("category")

Let's visualise the mapping in latent space. We will separate the query by day to show where they have been mapped relative to reference, and separately show the cell type annotations.

In [ ]:
#Compute UMAP
sc.pp.neighbors(adata_combined, use_rep="X_scVI")
sc.tl.umap(adata_combined)

adata_combined.obs["mapping_view"] = pd.Categorical(
    adata_combined.obs["mapping_view"],
    categories=["reference", "query" ] #query_day5", "query_day10"]
)

fig, axes = plt.subplots(2, 1, figsize=(17, 10))

# --------------------------
# UMAP 1: mapping view
# --------------------------
sc.pl.umap(
    adata_combined,
    color="mapping_view",
    palette={
        "reference": "lightgrey",
        "query": "red",
        "query_day5": "red",
        "query_day10": "blue",
    },
    ax=axes[0],
    show=False,
    title="Query mapped onto reference"
)

# --------------------------
# UMAP 2: combined labels
# --------------------------
sc.pl.umap(
    adata_combined,
    color="cell_type",
    ax=axes[1],
    show=False,
    title="Cell types (ref + predicted)"
)

plt.tight_layout()
plt.show()

In [ ]:
#Save the query data with the annotations

query_mask = adata_combined.obs["mapping_view"] == "query"
adata_query = adata_combined[query_mask].copy()


The final step is to quantify the cell type composition of our dataset.

In [ ]:
counts = adata_query.obs["cell_type"].value_counts()
df = counts.reset_index()
df.columns = ["cell_type", "number_cells"]
df["percentage_cells"] = df["number_cells"] / df["number_cells"].sum() * 100

Let's visualise the cell type distribution in a pie chart.

In [ ]:

# Make sure sorted (optional but nicer)
df = df.sort_values(by="number_cells", ascending=False)

sizes = df["number_cells"]
labels = df["cell_type"]
percentages = df["percentage_cells"]

# Function: only show % if >20%
def autopct_func(pct):
    return f"{pct:.1f}%" if pct > 20 else ""

# Build legend labels with %
legend_labels = [
    f"{ct} ({pct:.1f}%)"
    for ct, pct in zip(labels, percentages)
]

# Plot
fig, ax = plt.subplots(figsize=(7, 7))

wedges, texts, autotexts = ax.pie(
    sizes,
    autopct=autopct_func,
    startangle=90
)

# Equal aspect ratio = circle
ax.axis("equal")

# Add legend
ax.legend(
    wedges,
    legend_labels,
    title="Cell Types",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

plt.title("Cell Type Composition")
plt.tight_layout()
plt.show()

Calculate distance and entropy to get a confidence measure.

In [ ]:
X = adata_combined.obsm["X_scVI"]

ref_mask = adata_combined.obs["mapping_view"] == "reference"
query_mask = adata_combined.obs["mapping_view"] == "query"

X_ref = X[ref_mask]
X_query = X[query_mask]

labels_ref = adata_combined.obs.loc[ref_mask, "cell_type"].values

Distance:

In [ ]:
centroids = {}
for ct in np.unique(labels_ref):
    centroids[ct] = X_ref[labels_ref == ct].mean(axis=0)

cluster_names = list(centroids.keys())
centroid_matrix = np.vstack([centroids[ct] for ct in cluster_names])

from scipy.spatial.distance import cdist

dist_matrix = cdist(X_query, centroid_matrix, metric="euclidean")
# shape = (n_query_cells, n_clusters)

from scipy.special import softmax

probs = softmax(-dist_matrix, axis=1)

entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)

adata_combined.obs.loc[query_mask, "mapping_entropy"] = entropy

min_dist = dist_matrix.min(axis=1)

adata_combined.obs.loc[query_mask, "min_distance"] = min_dist

pred_labels = [cluster_names[i] for i in np.argmax(probs, axis=1)]

adata_combined.obs.loc[query_mask, "predicted_cell_type"] = pred_labels
adata_combined.obs.loc[query_mask, "max_prob"] = probs.max(axis=1)

In [ ]:
sc.pl.umap(adata_combined, color="mapping_entropy")

In [ ]:
#Entropy :
#< ~1.8: relatively confident
#~1.8–2.2: moderate confidence
#~2.2–2.5: fairly ambiguous
#> ~2.5: poor / diffuse match

Lastly, let's save the dataframe with the cell type quantification.

In [ ]:
df.to_csv("data/3802619/cell_type_quantification.csv", index=False)